# Dataset Builder — Cartes de perfusion + découpage en sous-clips AI-ready

## Ce que fait ce notebook

Pour chaque clip existant (`clip_1/`, `clip_2/`, ...) dans ta structure de données :

**Étape 1 — Cartes sur la totalité du clip (fenêtre longue)**  
Calcule 3 cartes de perfusion spatiale sur toute la durée du clip avec fenêtres glissantes :
- `snr_map` : SNR spectral en dB (Pulse3DFace Eq. 1–2)
- `corr_map` : Corrélation Pearson locale vs globale
- `corr_snr_map` : `corr × SNR_linéaire` (pondéré)

**Étape 2 — Découpage en sous-clips AI-ready**  
Découpe le clip en sous-clips de durée `d` secondes (stride configurable) :
```
clip_1/
  snr_map.npy          ← carte calculée sur tout le clip
  corr_map.npy
  corr_snr_map.npy
  subclip_0001/
    frames.npy         ← (T,H,W,3) uint8 RGB (resizé)
    masks.npy          ← (T,H,W) uint8 {0,255} (resizé)
    rppg_global.npy    ← signal rPPG global (T échantillons)
    rppg_global.png
  subclip_0002/
    ...
```

## Structure des données en entrée attendue
```
DATA_ROOT/
  S1/
    S1_4/
      clip_1/
        frames/          ← frames PNG numérotées
        masked_frames/   ← masks PNG correspondants
      clip_2/
        ...
```

In [13]:
import re
import math
import json
import shutil
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from typing import List, Tuple, Optional
from scipy import signal as scipy_signal
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

## ⚙️ Configuration — modifier ici

In [14]:
# ── Chemin vers les données (lecture) ────────────────────────────────────────
DATA_ROOT = Path(r"C:\Users\p02991\Desktop\Project rPPG face\data")

# ── Chemin de sortie (écriture des cartes + sous-clips) ───────────────────────
OUTPUT_ROOT = Path(r"C:\Users\p02991\Desktop\Project rPPG face\results")

# ── Caméra ───────────────────────────────────────────────────────────────────
fs = 30.0

# ── Fenêtre spatiale ─────────────────────────────────────────────────────────
k = 7

# ── Redimensionnement des sorties ─────────────────────────────────────────────
# Même méthode que extract_and_mask_v5.ipynb : cv2.INTER_AREA (downscaling).
# Appliqué aux cartes (snr_map, corr_map, corr_snr_map) ET aux frames/masques
# copiés dans les sous-clips.
# Mettre à None pour désactiver le redimensionnement.
RESIZE_TARGET_SIZE = (64, 64)   # (width, height) en pixels

# ── Sous-clips AI ────────────────────────────────────────────────────────────
subclip_duration_sec = 5.0
subclip_stride_sec   = 5.0

# ── Cartes (calculées sur tout le clip) ──────────────────────────────────────
map_window_sec  = 20.0
map_stride_sec  =  2.0

hr_min_bpm          = 30.0
hr_max_bpm          = 200.0
snr_delta_fund_bpm  =  6.0
snr_delta_harm_bpm  = 12.0

# ── POS Wang 2017 ────────────────────────────────────────────────────────────
pos_win_sec  = 1.6
pos_bp_low   = 0.70
pos_bp_high  = 3.0

# ── Misc ─────────────────────────────────────────────────────────────────────
min_skin_fraction = 0.5
pixel_chunk_size  = 30000
min_clip_sec      = 3.0
copy_frames       = True
overwrite         = False

# ── Traitement parallèle ─────────────────────────────────────────────────────
n_workers = 4

exts = ("png", "jpg", "jpeg", "bmp")


## POS Wang 2017

In [15]:
def pos_wang2017_raw(RGB: np.ndarray, fs: float, win_sec: float = 1.6) -> np.ndarray:
    """
    POS brut — Wang et al. 2017 IEEE TBME.
    RGB : (N, 3) float  →  H : (N,) float64 (signal accumulé, avant filtrage)
    """
    RGB = np.asarray(RGB, dtype=np.float64)
    N   = RGB.shape[0]
    l   = math.ceil(win_sec * fs)
    H   = np.zeros(N, dtype=np.float64)
    M   = np.array([[0., 1., -1.], [-2., 1., 1.]], dtype=np.float64)
    eps = 1e-12
    for n in range(l, N + 1):
        m   = n - l
        win = RGB[m:n, :]
        mu  = np.mean(win, axis=0, keepdims=True) + eps
        Cn  = (win / mu).T                          # (3, l)
        S   = M @ Cn                                # (2, l)
        s0, s1 = S[0], S[1]
        alpha  = (np.std(s0) + eps) / (np.std(s1) + eps)
        h      = s0 + alpha * s1
        H[m:n] += h - np.mean(h)
    return H


def pos_signal(RGB: np.ndarray, fs: float,
               win_sec: float = 1.6,
               bp_low: float  = 0.70,
               bp_high: float = 3.0) -> np.ndarray:
    """
    POS + détrend + bandpass + normalisation Hilbert.
    RGB : (N, 3)  →  (N,) float32
    """
    H   = pos_wang2017_raw(RGB, fs, win_sec)
    H   = scipy_signal.detrend(H, type="linear")
    nyq = fs / 2.0
    b, a = scipy_signal.butter(2, [bp_low / nyq, bp_high / nyq], btype="bandpass")
    H    = scipy_signal.filtfilt(b, a, H)
    env  = np.abs(scipy_signal.hilbert(H)) + 1e-12
    return (H / env).astype(np.float32)


def pos_local(RGB_tensor: np.ndarray, fs: float,
              win_sec: float = 1.6,
              bp_low: float  = 0.70,
              bp_high: float = 3.0) -> np.ndarray:
    """
    POS vectorisé par pixel.
    RGB_tensor : (N, H, W, 3)  →  (N, H, W) float32
    """
    RGB_tensor = np.asarray(RGB_tensor, dtype=np.float64)
    N, Hi, Wi, _ = RGB_tensor.shape
    l   = math.ceil(win_sec * fs)
    if N < l:
        return np.zeros((N, Hi, Wi), dtype=np.float32)
    M   = np.array([[0., 1., -1.], [-2., 1., 1.]], dtype=np.float64)
    Hb  = np.zeros((N, Hi, Wi), dtype=np.float64)
    eps = 1e-12
    for m in range(N - l + 1):
        win   = RGB_tensor[m:m+l]                               # (l, Hi, Wi, 3)
        mu    = np.mean(win, axis=0, keepdims=True) + eps
        Cn    = win / mu                                        # (l, Hi, Wi, 3)
        S     = np.einsum("ij,lhwj->ilhw", M, Cn)              # (2, l, Hi, Wi)
        s0, s1 = S[0], S[1]
        alpha = (np.std(s0, axis=0, keepdims=True) + eps) / \
                (np.std(s1, axis=0, keepdims=True) + eps)
        h     = s0 + alpha * s1
        Hb[m:m+l] += h - np.mean(h, axis=0, keepdims=True)
    Hb  = scipy_signal.detrend(Hb, axis=0, type="linear")
    nyq = fs / 2.0
    b, a = scipy_signal.butter(2, [bp_low / nyq, bp_high / nyq], btype="bandpass")
    Hb   = scipy_signal.filtfilt(b, a, Hb, axis=0)
    env  = np.abs(scipy_signal.hilbert(Hb, axis=0)) + eps
    return (Hb / env).astype(np.float32)


def hr_from_signal(sig: np.ndarray, fs: float,
                   hr_min: float = 30., hr_max: float = 200.) -> float:
    """Fréquence au max de puissance dans [hr_min, hr_max] BPM."""
    freqs = np.fft.rfftfreq(len(sig), d=1.0 / fs)
    pwr   = np.abs(np.fft.rfft(sig.astype(np.float64))) ** 2
    mask  = (freqs >= hr_min / 60.) & (freqs <= hr_max / 60.)
    if not np.any(mask):
        return 60.
    return float(freqs[mask][np.argmax(pwr[mask])] * 60.)

## Lecture des frames d'un clip

In [16]:
def list_images(folder: Path, exts: Tuple[str, ...]) -> List[Path]:
    files: List[Path] = []
    for e in exts:
        files.extend(folder.glob(f"*.{e}"))
    if not files:
        raise FileNotFoundError(f"No images in {folder}")
    return sorted(files, key=lambda p: p.name)


def find_mask(fp: Path, masks_dir: Path) -> Path:
    for e in ("png", "jpg", "jpeg", "bmp"):
        p = masks_dir / f"{fp.stem}.{e}"
        if p.exists():
            return p
    raise FileNotFoundError(f"No mask for {fp.name} in {masks_dir}")


def load_clip(frames_dir: Path, masks_dir: Path,
              k: int, min_skin_fraction: float,
              exts: Tuple[str, ...]) -> Tuple[
                  np.ndarray,   # local_rgb  (T, H, W, 3)
                  np.ndarray,   # global_rgb (T, 3)
                  np.ndarray,   # valid_mask (H, W) bool
                  List[Path],   # frame_files
                  List[Path]]:
    """
    Charge toutes les frames d'un clip.
    Retourne les signaux RGB locaux (fenêtre k×k) et globaux (peau entière).
    """
    frame_files = list_images(frames_dir, exts)
    T = len(frame_files)

    first = cv2.cvtColor(cv2.imread(str(frame_files[0])), cv2.COLOR_BGR2RGB)
    Hi, Wi = first.shape[:2]
    kernel = (k, k)

    # Masque de validité basé sur la 1ère frame
    m0   = cv2.imread(str(find_mask(frame_files[0], masks_dir)), cv2.IMREAD_GRAYSCALE)
    m0b  = (m0 > 0).astype(np.float32)
    sbox = cv2.boxFilter(m0b, -1, kernel, normalize=False)
    valid_mask = (sbox / (k * k)) >= min_skin_fraction  # (Hi, Wi)

    local_rgb  = np.zeros((T, Hi, Wi, 3), dtype=np.float32)
    global_rgb = np.zeros((T, 3),          dtype=np.float32)

    # Lire aussi les chemins de masks dans l'ordre (pour copie)
    mask_files = []
    for t, fp in enumerate(frame_files):
        frame    = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB).astype(np.float32)
        mfp      = find_mask(fp, masks_dir)
        mask_files.append(mfp)
        mask     = cv2.imread(str(mfp), cv2.IMREAD_GRAYSCALE)
        mask_bin = (mask > 0).astype(np.float32)
        skin_n   = np.sum(mask_bin)
        if skin_n > 0:
            for c in range(3):
                global_rgb[t, c] = np.sum(frame[:, :, c] * mask_bin) / skin_n
        sum_m = np.maximum(cv2.boxFilter(mask_bin, -1, kernel, normalize=False), 1e-6)
        for c in range(3):
            local_rgb[t, :, :, c] = (
                cv2.boxFilter(frame[:, :, c] * mask_bin, -1, kernel, normalize=False) / sum_m)

    return local_rgb, global_rgb, valid_mask, frame_files, mask_files

## Calcul des 3 cartes (SNR + corrélation + corrélation pondérée)

In [17]:
def compute_snr_map(local_rgb: np.ndarray,
                    s_ref: np.ndarray,
                    hr_ref_bpm: float,
                    fs: float,
                    valid_mask: np.ndarray,
                    hr_min_bpm: float, hr_max_bpm: float,
                    snr_delta_fund: float, snr_delta_harm: float,
                    pos_win_sec: float, pos_bp_low: float, pos_bp_high: float,
                    map_window_sec: float, map_stride_sec: float,
                    pixel_chunk_size: int) -> np.ndarray:
    """
    SNR map (Pulse3DFace Eq. 1-2) moyennée sur fenêtres glissantes.

    SNR(dB) = 10*log10( Σ_{f∈F_HR} (U_t(f)*|Ŝ(f)|²) /
                        Σ_{f∈F_HR} ((1-U_t(f))*|Ŝ(f)|²) )

    U_t(f) = 1 si |f - HRref| ≤ delta_fund OU |f - 2*HRref| ≤ delta_harm

    Retourne: (H, W) float32 — SNR en dB (NaN hors masque)
    """
    T, Hi, Wi, _ = local_rgb.shape
    win_len = int(round(map_window_sec * fs))
    stride  = int(round(map_stride_sec * fs))
    if win_len >= T:
        win_len = T
    starts = list(range(0, T - win_len + 1, stride)) or [0]

    valid_idx = np.flatnonzero(valid_mask.reshape(-1))
    snr_acc   = np.zeros(Hi * Wi, dtype=np.float64)
    count     = np.zeros(Hi * Wi, dtype=np.float64)

    for st in starts:
        en      = st + win_len
        seg     = local_rgb[st:en]  # (win, Hi, Wi, 3)
        T_seg   = seg.shape[0]
        freqs   = np.fft.rfftfreq(T_seg, d=1.0 / fs)
        hr_hz   = hr_ref_bpm / 60.
        dF      = snr_delta_fund / 60.
        dH      = snr_delta_harm / 60.

        # Masques fréquentiels
        f_hr = ((freqs >= hr_min_bpm / 60.) & (freqs <= hr_max_bpm / 60.)) | \
               ((freqs >= 2 * hr_min_bpm / 60.) & (freqs <= 2 * hr_max_bpm / 60.))
        U_t  = ((np.abs(freqs - hr_hz)       <= dF) |
                (np.abs(freqs - 2. * hr_hz)  <= dH)).astype(np.float32)

        rgb_flat = seg.reshape(T_seg, Hi * Wi, 3)
        M   = np.array([[0., 1., -1.], [-2., 1., 1.]])
        l   = math.ceil(pos_win_sec * fs)
        nyq = fs / 2.
        b, a = scipy_signal.butter(2, [pos_bp_low / nyq, pos_bp_high / nyq], btype="bandpass")
        eps = 1e-12

        for i0 in range(0, len(valid_idx), pixel_chunk_size):
            idx   = valid_idx[i0 : i0 + pixel_chunk_size]
            n_px  = len(idx)
            chunk = rgb_flat[:, idx, :]  # (T_seg, n_px, 3)

            # POS vectorisé
            Hb = np.zeros((T_seg, n_px), dtype=np.float64)
            for m in range(T_seg - l + 1):
                win = chunk[m:m+l]  # (l, n_px, 3)
                mu  = np.mean(win, axis=0, keepdims=True) + eps  # (1, n_px, 3)
                Cn  = (win / mu).transpose(2, 0, 1)              # (3, l, n_px)
                S   = np.tensordot(M, Cn, axes=([1], [0]))       # (2, l, n_px)
                s0, s1 = S[0], S[1]
                alpha  = (np.std(s0, axis=0, keepdims=True) + eps) / \
                         (np.std(s1, axis=0, keepdims=True) + eps)
                h   = s0 + alpha * s1
                Hb[m:m+l] += h - np.mean(h, axis=0, keepdims=True)

            Hb = scipy_signal.detrend(Hb, axis=0, type="linear")
            Hb = scipy_signal.filtfilt(b, a, Hb, axis=0)

            Shat  = np.abs(np.fft.rfft(Hb, axis=0)) ** 2  # (n_f, n_px)
            Sh_hr = Shat[f_hr, :]                           # (n_f_hr, n_px)
            U_hr  = U_t[f_hr]                               # (n_f_hr,)

            num = np.sum(U_hr[:, None]        * Sh_hr, axis=0)
            den = np.sum((1. - U_hr[:, None]) * Sh_hr, axis=0)
            snr_lin = num / np.maximum(den, eps)

            snr_acc[idx] += 10. * np.log10(snr_lin + eps)
            count[idx]   += 1.

    snr_flat = np.where(count > 0, snr_acc / np.maximum(count, 1), np.nan)
    snr_map  = snr_flat.reshape(Hi, Wi).astype(np.float32)
    snr_map  = np.where(valid_mask, snr_map, np.nan)
    return snr_map


def compute_corr_map(local_pos: np.ndarray,
                     s_ref: np.ndarray,
                     valid_mask: np.ndarray,
                     fs: float,
                     map_window_sec: float,
                     map_stride_sec: float,
                     pixel_chunk_size: int) -> np.ndarray:
    """
    Corrélation Pearson locale vs globale, moyennée sur fenêtres glissantes.
    local_pos : (T, H, W) float32
    s_ref     : (T,) float32
    Retourne  : (H, W) float32
    """
    T, Hi, Wi = local_pos.shape
    win_len   = int(round(map_window_sec * fs))
    stride    = int(round(map_stride_sec * fs))
    if win_len >= T:
        win_len = T
    starts    = list(range(0, T - win_len + 1, stride)) or [0]

    valid_idx = np.flatnonzero(valid_mask.reshape(-1))
    local_flat = local_pos.reshape(T, Hi * Wi)
    corr_acc   = np.zeros(Hi * Wi, dtype=np.float64)

    for st in starts:
        en    = st + win_len
        ref_w = s_ref[st:en].astype(np.float64)
        ref_z = (ref_w - ref_w.mean()) / (ref_w.std() + 1e-8)
        for i0 in range(0, len(valid_idx), pixel_chunk_size):
            idx  = valid_idx[i0 : i0 + pixel_chunk_size]
            loc  = local_flat[st:en, idx].T.astype(np.float64)  # (n_px, win)
            loc_z = (loc - loc.mean(axis=1, keepdims=True)) / \
                    (loc.std(axis=1,  keepdims=True) + 1e-8)
            corr_acc[idx] += (loc_z * ref_z[None, :]).mean(axis=1)

    corr_flat = (corr_acc / len(starts)).astype(np.float32)
    corr_map  = corr_flat.reshape(Hi, Wi)
    corr_map  = np.where(valid_mask, corr_map, np.nan)
    return corr_map


def compute_three_maps(local_rgb: np.ndarray,
                       global_rgb: np.ndarray,
                       valid_mask: np.ndarray,
                       fs: float,
                       pos_win_sec: float, pos_bp_low: float, pos_bp_high: float,
                       hr_min_bpm: float, hr_max_bpm: float,
                       snr_delta_fund: float, snr_delta_harm: float,
                       map_window_sec: float, map_stride_sec: float,
                       pixel_chunk_size: int):
    """
    Calcule snr_map, corr_map, corr_snr_map sur tout le clip.
    Retourne dict + hr_ref_bpm.
    """
    # Signal de référence global
    s_ref      = pos_signal(global_rgb, fs, pos_win_sec, pos_bp_low, pos_bp_high)
    hr_ref_bpm = hr_from_signal(s_ref, fs, hr_min_bpm, hr_max_bpm)

    # SNR map
    # print("    calcul SNR map...")
    snr_map = compute_snr_map(
        local_rgb, s_ref, hr_ref_bpm, fs, valid_mask,
        hr_min_bpm, hr_max_bpm, snr_delta_fund, snr_delta_harm,
        pos_win_sec, pos_bp_low, pos_bp_high,
        map_window_sec, map_stride_sec, pixel_chunk_size)

    # POS local pour corrélation
    # print("    calcul POS local...")
    local_pos = pos_local(local_rgb, fs, pos_win_sec, pos_bp_low, pos_bp_high)

    # print("    calcul corrélation...")
    corr_map  = compute_corr_map(
        local_pos, s_ref, valid_mask, fs,
        map_window_sec, map_stride_sec, pixel_chunk_size)

    corr_map = np.clip(corr_map, -1., 1.)

    # # SNR linéaire pour pondération
    # snr_lin     = 10. ** (snr_map / 10.)
    # corr_snr    = (corr_map * snr_lin).astype(np.float32)
    # corr_snr    = np.where(valid_mask, corr_snr, np.nan)


    snr_lin   = 10. ** (snr_map / 10.)
    # Normalise dans [0,1] par percentile robuste (évite les outliers)
    valid     = snr_lin[~np.isnan(snr_lin)]
    p2, p98   = np.percentile(valid, 2), np.percentile(valid, 98)
    snr_norm  = np.clip((snr_lin - p2) / (p98 - p2 + 1e-8), 0., 1.)
    corr_snr  = (corr_map * snr_norm).astype(np.float32)   # ∈ [-1, 1]
    corr_snr    = np.where(valid_mask, corr_snr, np.nan)

    return {
        "snr_map"     : snr_map,
        "corr_map"    : corr_map,
        "corr_snr_map": corr_snr,
        "hr_ref_bpm"  : hr_ref_bpm,
        "s_ref"       : s_ref,
    }

## Sauvegarde des cartes + visualisation

In [18]:
def _resize_map(arr: np.ndarray, target_size: tuple) -> np.ndarray:
    """
    Redimensionne une carte spatiale float32 (H, W) vers target_size.

    Même interpolation que extract_and_mask_v5.ipynb : cv2.INTER_AREA
    (optimal pour le downscaling).  Les valeurs NaN sont préservées :
    tout pixel de sortie provenant à >50 % de pixels NaN reste NaN.
    """
    tw, th = target_size
    if arr.shape == (th, tw):
        return arr
    nan_mask = np.isnan(arr)
    fill_val = float(np.nanmean(arr)) if not np.all(nan_mask) else 0.0
    arr_filled = np.where(nan_mask, fill_val, arr).astype(np.float32)
    resized    = cv2.resize(arr_filled, (tw, th), interpolation=cv2.INTER_AREA)
    nan_r      = cv2.resize(nan_mask.astype(np.float32), (tw, th),
                            interpolation=cv2.INTER_AREA)
    resized[nan_r > 0.5] = np.nan
    return resized.astype(np.float32)


def save_map(arr: np.ndarray, path_npy: Path, path_png: Path,
             cmap: str = "inferno", vmin=None, vmax=None,
             target_size: tuple = None):
    """
    Sauvegarde un array (H, W) en .npy + .png colorisé.
    Si target_size est fourni, redimensionne avant sauvegarde (cv2.INTER_AREA).
    """
    if target_size is not None:
        arr = _resize_map(arr, target_size)
    np.save(str(path_npy), arr)
    valid = arr[~np.isnan(arr)]
    lo = vmin if vmin is not None else (float(np.percentile(valid, 2))  if len(valid) else 0.)
    hi = vmax if vmax is not None else (float(np.percentile(valid, 98)) if len(valid) else 1.)
    if hi <= lo:
        hi = lo + 1e-6
    norm = np.clip((np.nan_to_num(arr, nan=lo) - lo) / (hi - lo), 0., 1.)
    rgba = matplotlib.colormaps[cmap](norm)
    rgba[np.isnan(arr)] = [0., 0., 0., 1.]
    img  = (rgba[:, :, :3] * 255).astype(np.uint8)
    cv2.imwrite(str(path_png), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))


def save_rppg_signal(signal: np.ndarray, out_dir: Path, fs: float):
    """Sauvegarde le signal rPPG en .npy + .png (courbe)."""
    np.save(str(out_dir / "rppg_global.npy"), signal.astype(np.float32))
    t = np.arange(len(signal)) / fs
    fig, ax = plt.subplots(figsize=(6, 1.5))
    ax.plot(t, signal, linewidth=0.8, color="#2ecc71")
    ax.set_xlabel("time (s)", fontsize=7)
    ax.set_ylabel("rPPG",     fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(str(out_dir / "rppg_global.png"), dpi=80)
    plt.close(fig)


## Découpage en sous-clips

In [19]:
def create_subclips(clip_dir: Path,
                    out_clip_dir: Path,
                    frame_files: List[Path],
                    mask_files:  List[Path],
                    s_ref_full:  np.ndarray,
                    hr_ref_bpm:  float,
                    fs: float,
                    subclip_duration_sec: float,
                    subclip_stride_sec:  float,
                    copy_frames: bool,
                    overwrite:   bool,
                    target_size: tuple = None) -> int:
    """
    Découpe le clip en sous-clips de durée fixe.

    clip_dir     : répertoire source (lecture des frames)
    out_clip_dir : répertoire de sortie (écriture des sous-clips)
    target_size  : (width, height) ou None — si fourni, redimensionne frames
                   et masques avec cv2.INTER_AREA (même méthode qu'extract_and_mask_v5).

    Retourne le nombre de sous-clips créés.
    """
    T         = len(frame_files)
    n_frames  = int(round(subclip_duration_sec * fs))
    stride    = int(round(subclip_stride_sec   * fs))

    if n_frames > T:
        n_frames = T
    if stride < 1:
        stride = 1

    starts = list(range(0, T - n_frames + 1, stride))
    if not starts:
        return 0

    out_clip_dir.mkdir(parents=True, exist_ok=True)

    n_created = 0
    for i, t_start in enumerate(starts):
        t_end  = t_start + n_frames
        sc_dir = out_clip_dir / f"subclip_{i+1:04d}"

        if (not overwrite
            and (sc_dir / "frames.npy").exists()
            and (sc_dir / "masks.npy").exists()
            and (sc_dir / "rppg_global.npy").exists()):
            continue

        sc_dir.mkdir(exist_ok=True)

        # ── Sauvegarde (npy) frames + masques, resize optionnel ────────────
        # frames.npy : (T, H, W, 3) uint8 en RGB
        # masks.npy  : (T, H, W) uint8 en {0,255}
        imgs  = []
        masks = []
        for global_idx in range(t_start, t_end):
            src_f = frame_files[global_idx]
            src_m = mask_files[global_idx]

            img_bgr = cv2.imread(str(src_f), cv2.IMREAD_COLOR)
            if img_bgr is None:
                raise RuntimeError(f"Impossible de lire frame: {src_f}")
            if target_size is not None:
                img_bgr = cv2.resize(img_bgr, target_size, interpolation=cv2.INTER_AREA)
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            imgs.append(img_rgb)

            m = cv2.imread(str(src_m), cv2.IMREAD_GRAYSCALE)
            if m is None:
                raise RuntimeError(f"Impossible de lire masque: {src_m}")
            if target_size is not None:
                m = cv2.resize(m, target_size, interpolation=cv2.INTER_AREA)
            m = (m > 127).astype(np.uint8) * 255
            masks.append(m)

        frames_arr = np.stack(imgs, axis=0).astype(np.uint8)
        masks_arr  = np.stack(masks, axis=0).astype(np.uint8)
        np.save(str(sc_dir / "frames.npy"), frames_arr)
        np.save(str(sc_dir / "masks.npy"),  masks_arr)

        # ── Signal rPPG global (on garde .npy + .png) ─────────────────────
        rppg_seg = s_ref_full[t_start:t_end].astype(np.float32)
        save_rppg_signal(rppg_seg, sc_dir, fs)

        n_created += 1

    return n_created


## Navigation dans la structure de données

In [20]:
def find_sessions(data_root: Path) -> List[Path]:
    subj_re = re.compile(r"^S\d+$")
    sess_re = re.compile(r"^S\d+_\d+$")
    sessions = []
    for subj in sorted(data_root.iterdir()):
        if not subj.is_dir() or not subj_re.match(subj.name):
            continue
        for sess in sorted(subj.iterdir()):
            if not sess.is_dir() or not sess_re.match(sess.name):
                continue
            sessions.append(sess)
    return sessions


def get_clip_dirs(session_dir: Path) -> List[Path]:
    clips = []
    for d in session_dir.iterdir():
        if not d.is_dir() or not d.name.startswith("clip_"):
            continue
        if (d / "frames").is_dir() and (d / "masked_frames").is_dir():
            try:
                clips.append((int(d.name.split("_")[1]), d))
            except (IndexError, ValueError):
                pass
    return [d for _, d in sorted(clips)]

## 🚀 Boucle principale

In [ ]:
sessions    = find_sessions(DATA_ROOT)
print(f"{len(sessions)} sessions trouvées\n")
print(f"DATA_ROOT        : {DATA_ROOT}")
print(f"OUTPUT_ROOT      : {OUTPUT_ROOT}")
print(f"RESIZE_TARGET_SIZE : {RESIZE_TARGET_SIZE}  (None = pas de resize)")
print(f"Paramètres sous-clips : durée={subclip_duration_sec}s | stride={subclip_stride_sec}s")
print(f"Paramètres cartes     : fenêtre={map_window_sec}s | stride={map_stride_sec}s")
print(f"copy_frames={copy_frames} | overwrite={overwrite} | n_workers={n_workers}")
print("=" * 65)


def _delete_frame_dirs(clip_dir: Path) -> int:
    deleted = 0
    for subc_dir in sorted(clip_dir.glob("subclip_*/")):
        for dir_name in ("frames", "masked_frames"):
            to_del = subc_dir / dir_name
            if to_del.exists():
                shutil.rmtree(to_del)
                deleted += 1
    return deleted


def process_one_clip(sess: Path, clip_dir: Path):
    """Traite un clip (cartes + sous-clips). Retourne (label, n_subclips, error_msg)."""
    label        = f"{sess.name}/{clip_dir.name}"
    frames_dir   = clip_dir / "frames"
    masks_dir    = clip_dir / "masked_frames"
    out_clip_dir = OUTPUT_ROOT / clip_dir.relative_to(DATA_ROOT)

    try:
        frame_files = list_images(frames_dir, exts)
        T = len(frame_files)
        if T / fs < min_clip_sec:
            return (label, "skip_short", None)

        local_rgb, global_rgb, valid_mask, frame_files, mask_files = load_clip(
            frames_dir, masks_dir, k, min_skin_fraction, exts)

        maps_done     = (out_clip_dir / "snr_map.npy").exists()
        sc1 = out_clip_dir / "subclip_0001"
        subclips_done = (sc1 / "frames.npy").exists() and (sc1 / "masks.npy").exists() and (sc1 / "rppg_global.npy").exists()
        if not overwrite and maps_done and subclips_done:
            return (label, "skip_done", None)

        out_clip_dir.mkdir(parents=True, exist_ok=True)

        if overwrite or not maps_done:
            result = compute_three_maps(
                local_rgb, global_rgb, valid_mask, fs,
                pos_win_sec, pos_bp_low, pos_bp_high,
                hr_min_bpm, hr_max_bpm,
                snr_delta_fund_bpm, snr_delta_harm_bpm,
                map_window_sec, map_stride_sec, pixel_chunk_size)
            # target_size applique cv2.INTER_AREA avant sauvegarde (même méthode
            # que _crop_and_resize_single_pass dans extract_and_mask_v5.ipynb)
            save_map(result["snr_map"],      out_clip_dir / "snr_map.npy",      out_clip_dir / "snr_map.png",      cmap="inferno", vmin=-10, vmax=0,  target_size=RESIZE_TARGET_SIZE)
            save_map(result["corr_map"],     out_clip_dir / "corr_map.npy",     out_clip_dir / "corr_map.png",     cmap="RdBu",    vmin=-1,  vmax=1,  target_size=RESIZE_TARGET_SIZE)
            save_map(result["corr_snr_map"], out_clip_dir / "corr_snr_map.npy", out_clip_dir / "corr_snr_map.png", cmap="inferno",                    target_size=RESIZE_TARGET_SIZE)
            s_ref, hr_ref_bpm = result["s_ref"], result["hr_ref_bpm"]
        else:
            s_ref      = pos_signal(global_rgb, fs, pos_win_sec, pos_bp_low, pos_bp_high)
            hr_ref_bpm = hr_from_signal(s_ref, fs, hr_min_bpm, hr_max_bpm)

        n_sc = create_subclips(
            clip_dir=clip_dir,
            out_clip_dir=out_clip_dir,
            frame_files=frame_files,
            mask_files=mask_files,
            s_ref_full=s_ref,
            hr_ref_bpm=hr_ref_bpm,
            fs=fs,
            subclip_duration_sec=subclip_duration_sec,
            subclip_stride_sec=subclip_stride_sec,
            copy_frames=copy_frames,
            overwrite=overwrite,
            target_size=RESIZE_TARGET_SIZE,
        )

        # plus de copie d'images dans les sous-clips (npy uniquement)

        return (label, n_sc, None)
    except Exception as e:
        return (label, None, str(e))


tasks = []
for sess in sessions:
    clip_dirs = get_clip_dirs(sess)
    if not clip_dirs:
        continue
    for clip_dir in clip_dirs:
        label        = f"{sess.name}/{clip_dir.name}"
        out_clip_dir = OUTPUT_ROOT / clip_dir.relative_to(DATA_ROOT)
        maps_done     = (out_clip_dir / "snr_map.npy").exists()
        sc1 = out_clip_dir / "subclip_0001"
        subclips_done = (sc1 / "frames.npy").exists() and (sc1 / "masks.npy").exists() and (sc1 / "rppg_global.npy").exists()
        if not overwrite and maps_done and subclips_done:
            print(f"SKIP {label}")
            continue
        tasks.append((sess, clip_dir))

total_clips    = 0
total_subclips = 0
errors         = []
results_lock   = threading.Lock()

with ThreadPoolExecutor(max_workers=n_workers) as executor:
    futures = {executor.submit(process_one_clip, sess, clip_dir): (sess, clip_dir)
               for sess, clip_dir in tasks}
    for future in as_completed(futures):
        label, n_sc, err = future.result()
        with results_lock:
            if err is not None:
                msg = f"ERR {label}: {err}"
                print(msg)
                errors.append(msg)
            elif n_sc in ("skip_short", "skip_done"):
                if n_sc == "skip_short":
                    print(f"SKIP {label}: trop court")
            else:
                print(f"  OK {label} | {n_sc} sous-clips")
                total_clips    += 1
                total_subclips += n_sc

print(f"\n{'='*65}")
print(f"Terminé : {total_clips} clips | {total_subclips} sous-clips")
print(f"Résultats dans : {OUTPUT_ROOT}")
if errors:
    print(f"Erreurs ({len(errors)}):")
    for e in errors:
        print(" ", e)


344 sessions trouvées

DATA_ROOT        : C:\Users\p02991\Desktop\Project rPPG face\data
OUTPUT_ROOT      : C:\Users\p02991\Desktop\Project rPPG face\results
RESIZE_TARGET_SIZE : (64, 64)  (None = pas de resize)
Paramètres sous-clips : durée=5.0s | stride=5.0s
Paramètres cartes     : fenêtre=20.0s | stride=2.0s
copy_frames=True | overwrite=False | n_workers=4
  OK S1_1/clip_1 | 4 sous-clips
  OK S1_1/clip_3 | 4 sous-clips
  OK S1_10/clip_1 | 4 sous-clips
  OK S1_1/clip_2 | 4 sous-clips
  OK S1_11/clip_1 | 4 sous-clips
  OK S1_11/clip_2 | 4 sous-clips
  OK S1_10/clip_2 | 4 sous-clips
  OK S1_10/clip_3 | 4 sous-clips
  OK S1_11/clip_3 | 4 sous-clips
  OK S1_12/clip_1 | 4 sous-clips
  OK S1_12/clip_2 | 4 sous-clips
  OK S1_12/clip_3 | 4 sous-clips
  OK S1_13/clip_1 | 4 sous-clips
  OK S1_13/clip_2 | 4 sous-clips
  OK S1_13/clip_3 | 4 sous-clips
  OK S1_14/clip_1 | 4 sous-clips
  OK S1_14/clip_2 | 4 sous-clips
  OK S1_15/clip_1 | 4 sous-clips
  OK S1_15/clip_2 | 4 sous-clips
  OK S1_15/cli

## 📊 Vérification — afficher le contenu d'un clip

Cellule optionnelle pour inspecter les cartes et un sous-clip.

In [ ]:
# Changer ce chemin pour inspecter un clip
check_clip = DATA_ROOT / "S1" / "S1_1" / "clip_1"

if not check_clip.exists():
    print(f"Clip non trouvé : {check_clip}")
else:
    # ── Cartes ──────────────────────────────────────────────────────────────
    maps_check = {}
    for nm, label, cmap, vmin, vmax in [
        ("snr_map",      "SNR (dB)",       "inferno", -10,  0),
        ("corr_map",     "Corrélation",    "RdBu",    -1,   1),
        ("corr_snr_map", "Corr × SNR",    "inferno",  None, None),
    ]:
        p = check_clip / f"{nm}.npy"
        if p.exists():
            maps_check[nm] = (np.load(str(p)), label, cmap, vmin, vmax)

    if maps_check:
        fig, axes = plt.subplots(1, len(maps_check), figsize=(5 * len(maps_check), 4))
        if len(maps_check) == 1:
            axes = [axes]
        for ax, (nm, (arr, label, cmap, vmin, vmax)) in zip(axes, maps_check.items()):
            im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
            ax.set_title(label)
            ax.axis("off")
            plt.colorbar(im, ax=ax, fraction=0.046)
        plt.suptitle(str(check_clip.relative_to(DATA_ROOT)))
        plt.tight_layout()
        plt.show()
    else:
        print("Aucune carte trouvée — lancer d'abord la boucle principale.")

    # ── Premier sous-clip ────────────────────────────────────────────────────
    sc = check_clip / "subclip_0001"
    if sc.exists():
        fp = sc / "frames.npy"
        mp = sc / "masks.npy"
        if fp.exists() and mp.exists():
            frames = np.load(str(fp))
            masks  = np.load(str(mp))
            rp = sc / "rppg_global.npy"
            rppg = np.load(str(rp)) if rp.exists() else None
            print(f"subclip_0001: frames {frames.shape} {frames.dtype} | masks {masks.shape} {masks.dtype} | rppg {None if rppg is None else (rppg.shape, rppg.dtype)}")

            if rppg is not None:
                t = np.arange(len(rppg)) / fs
                fig, ax = plt.subplots(figsize=(8, 2))
                ax.plot(t, rppg, linewidth=0.9, color="#2ecc71")
                ax.set_xlabel("time (s)")
                ax.set_ylabel("rPPG (norm.)")
                ax.set_title("subclip_0001 — rppg_global.npy")
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()

            # Affiche 3 exemples (frame + masque)
            n_show = min(3, frames.shape[0])
            fig, axes = plt.subplots(n_show, 2, figsize=(6, 3 * n_show))
            if n_show == 1:
                axes = np.array([axes])
            for j in range(n_show):
                axes[j, 0].imshow(frames[j])
                axes[j, 0].set_title(f"frame[{j}]")
                axes[j, 0].axis("off")
                axes[j, 1].imshow(masks[j], cmap="gray", vmin=0, vmax=255)
                axes[j, 1].set_title(f"mask[{j}]")
                axes[j, 1].axis("off")
            plt.tight_layout()
            plt.show()

            # Liste des sous-clips
            scs = sorted(check_clip.glob("subclip_*/frames.npy"))
            print(f"\n{len(scs)} sous-clips dans ce clip (frames.npy trouvés) :")
            for p in scs[:5]:
                arr = np.load(str(p), mmap_mode="r")
                print(f"  {p.parent.name}: {tuple(arr.shape)}")
            if len(scs) > 5:
                print(f"  ... et {len(scs)-5} autres")
        else:
            print("subclip_0001 existe mais frames.npy/masks.npy manquent.")
    else:
        print("Aucun sous-clip trouvé — lancer d'abord la boucle principale.")

Aucune carte trouvée — lancer d'abord la boucle principale.
Aucun sous-clip trouvé — lancer d'abord la boucle principale.


## 📋 Résumé du dataset

Cellule pour avoir un bilan global de tous les sous-clips créés.

In [ ]:
sessions = find_sessions(DATA_ROOT)
all_subclips = []

for sess in sessions:
    for clip_dir in get_clip_dirs(sess):
        for frames_p in sorted(clip_dir.glob("subclip_*/frames.npy")):
            masks_p = frames_p.parent / "masks.npy"
            rppg_p  = frames_p.parent / "rppg_global.npy"
            if (not masks_p.exists()) or (not rppg_p.exists()):
                continue
            arr = np.load(str(frames_p), mmap_mode="r")
            all_subclips.append({
                "session": sess.name,
                "clip": clip_dir.name,
                "subclip": frames_p.parent.name,
                "n_frames": int(arr.shape[0]),
                "h": int(arr.shape[1]),
                "w": int(arr.shape[2]),
                "c": int(arr.shape[3]) if len(arr.shape) == 4 else None,
            })

print(f"Total sous-clips  : {len(all_subclips)}")
if all_subclips:
    n_frames = [s["n_frames"] for s in all_subclips]
    shapes = sorted(set((s["h"], s["w"], s["c"]) for s in all_subclips))
    sessions_list = sorted(set(s["session"] for s in all_subclips))
    print(f"Frames / sous-clip: {n_frames[0]} (fixe si stride/durée fixes)")
    print(f"Tailles (HxWxC)   : {shapes}")
    print(f"Sessions          : {len(sessions_list)} → {sessions_list}")

Total sous-clips  : 0
